# Agentic AI Patterns: Tool Use & Planning

## Tool Use & Function Calling

Tool use enables LLMs to call external functions or APIs. The agent decides which tool to use, the LLM generates structured tool calls, and the system executes them. This loop continues until the agent reaches a final answer. The pattern: $\text{Agent} \to \text{Tool Call} \to \text{Execution} \to \text{Result} \to \text{Agent}$.

```python title="example1.py"
import json
from typing import Any
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.llms import HuggingFacePipeline
from langchain.tools import tool

# Define tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression"""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    # Simulated search result
    return f"Search results for '{query}': [Mock result 1, Mock result 2]"

@tool
def get_current_time() -> str:
    """Get current date and time"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Create agent with tools
tools = [calculator, search_web, get_current_time]
llm = HuggingFacePipeline(model_name="gpt2")

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
result = agent.run("What is 25 * 4? Also tell me the current time.")
print(result)
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-agents/mod-25.ipynb)

```
Thought: I need to calculate 25 * 4 and get the current time.
Action: calculator
Action Input: 25 * 4
Observation: 100
Action: get_current_time
Action Input: 
Observation: 2024-05-24 17:50:46
Final Answer: 25 * 4 equals 100. The current time is 2024-05-24 17:50:46.
```

## Planning & Task Decomposition

Planning breaks complex tasks into subtasks. The agent generates a plan, executes steps sequentially, and adapts if needed. This enables solving problems that require multiple steps and tool interactions.

```python title="example2.py"
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate

# Planning prompt
planning_prompt = PromptTemplate(
    input_variables=["task"],
    template="""Break down this task into clear steps:
Task: {task}

Steps:
1. 
2. 
3. 
"""
)

# Define subtask tools
def research_step(topic: str) -> str:
    """Research a topic"""
    return f"Research findings on {topic}: [Key points...]"

def analyze_step(data: str) -> str:
    """Analyze data"""
    return f"Analysis of {data}: [Insights...]"

def summarize_step(findings: str) -> str:
    """Summarize findings"""
    return f"Summary: {findings}"

# Create planning agent
tools = [
    Tool(name="Research", func=research_step, description="Research a topic"),
    Tool(name="Analyze", func=analyze_step, description="Analyze data"),
    Tool(name="Summarize", func=summarize_step, description="Summarize findings")
]

llm = HuggingFacePipeline(model_name="gpt2")
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Execute multi-step task
task = "Research machine learning trends, analyze their impact, and summarize findings"
result = agent.run(task)
print(result)
```

> **💡 Tip:** Use explicit planning prompts to guide agents. Break tasks into 3-5 steps for better performance.

## Reflection & Self-Correction

Reflection enables agents to evaluate their outputs and correct mistakes. The pattern: $\text{Generate} \to \text{Evaluate} \to \text{Reflect} \to \text{Correct}$.

```python title="example3.py"
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate

# Generation prompt
generation_prompt = PromptTemplate(
    input_variables=["task"],
    template="Complete this task: {task}\nAnswer:"
)

# Evaluation prompt
evaluation_prompt = PromptTemplate(
    input_variables=["task", "answer"],
    template="""Evaluate this answer for the task: {task}
Answer: {answer}

Is this answer correct? (Yes/No)
What could be improved?
"""
)

# Correction prompt
correction_prompt = PromptTemplate(
    input_variables=["task", "answer", "feedback"],
    template="""Original task: {task}
Previous answer: {answer}
Feedback: {feedback}

Provide a corrected answer:
"""
)

class ReflectiveAgent:
    def __init__(self, llm):
        self.llm = llm
    
    def run(self, task: str, max_iterations: int = 3):
        answer = self.llm(generation_prompt.format(task=task))
        
        for i in range(max_iterations):
            # Evaluate
            evaluation = self.llm(
                evaluation_prompt.format(task=task, answer=answer)
            )
            
            if "Yes" in evaluation or "correct" in evaluation.lower():
                return answer
            
            # Reflect and correct
            answer = self.llm(
                correction_prompt.format(
                    task=task,
                    answer=answer,
                    feedback=evaluation
                )
            )
        
        return answer

# Use reflective agent
llm = HuggingFacePipeline(model_name="gpt2")
agent = ReflectiveAgent(llm)
result = agent.run("Explain quantum entanglement in simple terms")
print(result)
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary purpose of tool use in AI agents?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500001" value="0">
      <span>To increase model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500001" value="1">
      <span>To enable agents to interact with external systems and APIs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500001" value="2">
      <span>To reduce inference latency</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500001" value="3">
      <span>To improve tokenization</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is the benefit of planning in agentic systems?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500002" value="0">
      <span>Reduces model parameters</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500002" value="1">
      <span>Improves tokenization accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500002" value="2">
      <span>Breaks complex tasks into manageable steps for better execution</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387500002" value="3">
      <span>Increases inference speed</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>